# Validación y métrica del negocio del proyecto de Ecobici

El modelo final (gradient boosting) y el segundo mejor (la red neuronal), con validación cruzada en dos esquemas: el temporal, que corresponde a un modelo que predice el futuro, y uno al azar para comparar. Guarda `validacion_proyecto.csv` con el promedio, la desviación y cada pliegue.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_predict, cross_validate
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline

import sys

sys.path.insert(0, "../../semana-09-sesion-02/plantilla-pipeline")
from pipeline_proyecto import RUTA_DATOS, cargar_y_dividir, construir_preparacion

X_ent, X_prueba, y_ent, y_prueba = cargar_y_dividir()

# Para validar, todo agosto junto y ordenado en el tiempo: la validacion
# temporal necesita que cada pliegue vea solo el pasado.
fechas = pd.read_csv(RUTA_DATOS, usecols=["fecha", "franja"]).loc[pd.concat([X_ent, X_prueba]).index]
orden = (fechas["fecha"] + " " + fechas["franja"].astype(str).str.zfill(2)).sort_values(kind="stable").index
X = pd.concat([X_ent, X_prueba]).loc[orden]
y = pd.concat([y_ent, y_prueba]).loc[orden]
print("Filas:", X.shape, "| desde", fechas.loc[orden[0], "fecha"], "hasta", fechas.loc[orden[-1], "fecha"])

Filas: (297500, 11) | desde 2026-08-01 hasta 2026-08-31


## Los dos modelos y la función que valida

In [2]:
def gradient_boosting():
    return GradientBoostingRegressor(random_state=42)


def red_neuronal():
    return MLPRegressor(hidden_layer_sizes=(64,), max_iter=200, early_stopping=True, random_state=42)


resultados = {}


def evaluar(nombre, modelo, esquema):
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    inicio = time.perf_counter()
    puntajes = cross_validate(pipeline, X, y, cv=esquema, scoring="neg_mean_absolute_error")
    errores = -puntajes["test_score"]
    resultados[nombre] = {
        "validacion": nombre,
        "mae_promedio": round(errores.mean(), 4),
        "desviacion": round(errores.std(), 4),
        "pliegues": [round(error, 4) for error in errores],
    }
    print(f"{nombre:32s} promedio {errores.mean():.4f}  desviacion {errores.std():.4f}  "
          f"pliegues {resultados[nombre]['pliegues']}  ({time.perf_counter() - inicio:.0f} s)")

## El esquema que corresponde: temporal

Cinco pliegues en orden de tiempo: cada uno entrena con los días anteriores y valida con los siguientes, como el modelo en producción.

In [3]:
temporal = TimeSeriesSplit(n_splits=5)
evaluar("temporal, gradient boosting", gradient_boosting(), temporal)
evaluar("temporal, red neuronal", red_neuronal(), temporal)

temporal, gradient boosting      promedio 1.0000  desviacion 0.0369  pliegues [0.952, 1.0275, 1.054, 0.9715, 0.9951]  (52 s)


temporal, red neuronal           promedio 0.9930  desviacion 0.0322  pliegues [0.9621, 1.0219, 1.0406, 0.964, 0.9764]  (258 s)


## Para comparar: al azar

In [4]:
al_azar = KFold(n_splits=5, shuffle=True, random_state=42)
evaluar("al azar, gradient boosting", gradient_boosting(), al_azar)
evaluar("al azar, red neuronal", red_neuronal(), al_azar)

al azar, gradient boosting       promedio 0.9904  desviacion 0.0022  pliegues [0.9898, 0.988, 0.9886, 0.994, 0.9917]  (89 s)


al azar, red neuronal            promedio 0.9790  desviacion 0.0053  pliegues [0.9795, 0.9693, 0.9792, 0.9848, 0.9824]  (266 s)


## La tabla de validación

In [5]:
tabla = pd.DataFrame(resultados.values())
tabla.to_csv("validacion_proyecto.csv", index=False)
tabla

,validacion,mae_promedio,desviacion,pliegues
0,"temporal, gradient boosting",1.0000,0.0369,"[0.952, 1.0275, 1.054, 0.9715, 0.9951]"
1,"temporal, red neuronal",0.9930,0.0322,"[0.9621, 1.0219, 1.0406, 0.964, 0.9764]"
2,"al azar, gradient boosting",0.9904,0.0022,"[0.9898, 0.988, 0.9886, 0.994, 0.9917]"
3,"al azar, red neuronal",0.9790,0.0053,"[0.9795, 0.9693, 0.9792, 0.9848, 0.9824]"


## La métrica del negocio

Subestimar las salidas es el error caro: si el modelo dice que saldrán pocas bicis y salen muchas, la estación se vacía sin que el camión de rebalanceo llegue, y cada bici que faltó es un viaje perdido (un usuario que se va a pie, en taxi o con otro servicio). Sobreestimar cuesta menos: un viaje del camión que no hacía falta, repartido entre las varias bicis que mueve. Supuesto aproximado: **una bici subestimada cuesta 3 veces lo que una sobreestimada**.

La métrica es el costo promedio por franja: 3 por cada bici subestimada y 1 por cada bici sobreestimada. Además, la proporción de franjas con demanda alta (3 salidas o más en 30 minutos) en las que el modelo se quedó corto por 2 bicis o más: las que terminan en una estación vacía.

In [6]:
COSTO_SUBESTIMAR = 3
COSTO_SOBREESTIMAR = 1


def costo_asimetrico(real, estimada):
    diferencia = np.asarray(real) - np.asarray(estimada)
    return np.mean(np.where(diferencia > 0, COSTO_SUBESTIMAR * diferencia, -COSTO_SOBREESTIMAR * diferencia))


def cortas_en_demanda_alta(real, estimada):
    real, estimada = np.asarray(real), np.asarray(estimada)
    alta = real >= 3
    return np.mean(real[alta] - estimada[alta] >= 2)


def persistencia(tabla):
    return tabla["salidas"] + tabla["salidas_lag_15m"].fillna(0)


metricas = []
estimaciones = {"persistencia": persistencia(X)}
for nombre, modelo in [("gradient boosting", gradient_boosting()), ("red neuronal", red_neuronal())]:
    pipeline = Pipeline([("preparar", construir_preparacion()), ("modelo", modelo)])
    # Validacion temporal: el primer bloque solo entrena, asi que se mide sobre los otros cuatro.
    estimada = pd.Series(np.nan, index=X.index)
    for entrenar, validar in temporal.split(X):
        pipeline.fit(X.iloc[entrenar], y.iloc[entrenar])
        estimada.iloc[validar] = pipeline.predict(X.iloc[validar])
    estimaciones[nombre] = estimada

validadas = estimaciones["gradient boosting"].notna()
for nombre, estimada in estimaciones.items():
    real, est = y[validadas], estimada[validadas]
    metricas.append({
        "modelo": nombre,
        "mae": round(mean_absolute_error(real, est), 4),
        "costo_asimetrico": round(costo_asimetrico(real, est), 4),
        "subestimadas": round(np.mean(real > est), 4),
        "cortas_en_demanda_alta": round(cortas_en_demanda_alta(real, est), 4),
    })
metricas_negocio = pd.DataFrame(metricas)
metricas_negocio.to_csv("metricas_negocio_proyecto.csv", index=False)
metricas_negocio

,modelo,mae,costo_asimetrico,subestimadas,cortas_en_demanda_alta
0,persistencia,1.1767,2.3534,0.2737,0.5193
1,gradient boosting,1.0000,1.9993,0.3899,0.3913
2,red neuronal,0.9930,1.9889,0.3865,0.3919
